In [1]:
import requests
import pandas as pd
from pathlib import Path
import os

In [2]:
headers = {"User-Agent": "Zain zain@unicusresearch.com"}
headers_submissions = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

# Ticker of Companies on SEC

In [3]:
url_sec_companies = "https://www.sec.gov/files/company_tickers.json"
resp_sec_companies = requests.get(url=url_sec_companies, headers=headers)

resp_sec_companies.status_code

200

In [4]:
df_sec_companies = pd.DataFrame(resp_sec_companies.json().values())
df_sec_companies

,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC
...,...,...,...
10421,2083217,QADRW,QDRO Acquisition Corp.
10422,2083632,OCTLF,Octave Intelligence plc
10423,2083217,QADRU,QDRO Acquisition Corp.
10424,2080023,VACI-UN,Viking Acquisition Corp I


In [5]:
company_ticker = "OKLO"
company_cik = str(list(df_sec_companies[df_sec_companies["ticker"]==company_ticker]["cik_str"])[0]).zfill(10)
company_cik

'0001849056'

# Ticker of Funds on SEC
Non traded funds like CCLFX registered with SEC but not listed on any exchange, so they won't appear in either ticker mapping file and have to directly find their cik via edgar

In [6]:
url_sec_funds = "https://www.sec.gov/files/company_tickers_mf.json"
resp_sec_funds = requests.get(url_sec_funds, headers=headers)

resp_sec_funds.status_code

200

In [7]:
json_sec_funds = resp_sec_funds.json()
df_sec_funds = pd.DataFrame(json_sec_funds["data"], columns=json_sec_funds["fields"])
df_sec_funds

,cik,seriesId,classId,symbol
0,2110,S000009184,C000024954,LACAX
1,2110,S000009184,C000024956,LIACX
2,2110,S000009184,C000024957,ACRNX
3,2110,S000009184,C000122736,CRBRX
4,2110,S000009184,C000122737,CRBYX
...,...,...,...,...
28396,2081199,S000105287,C000276035,BROL
28397,2083193,S000097445,C000266630,FEMD
28398,2083193,S000097446,C000266631,USFE
28399,2096904,S000101078,C000271016,MHIG


In [8]:
fund_ticker = "CCLFX"
fund_cik = str(df_sec_funds[df_sec_funds["symbol"]==fund_ticker]["cik"].values[0]).zfill(10)
fund_cik

IndexError: index 0 is out of bounds for axis 0 with size 0

In [9]:
fund_ticker = "CCLFX"
fund_cik = str(df_sec_funds[df_sec_funds["symbol"]==fund_ticker])
fund_cik

'Empty DataFrame\nColumns: [cik, seriesId, classId, symbol]\nIndex: []'

# Submissions by a company

In [10]:
# company_cik = "0001658645"
# company_cik = "0001735964"
company_cik = '0001842754'

In [11]:
url_submissions = f"https://data.sec.gov/submissions/CIK{company_cik}.json"
resp_submissions = requests.get(url_submissions, headers=headers)
print(resp_submissions.status_code)

json_submissions = resp_submissions.json()

200


In [12]:
json_submissions

{'cik': '0001842754',
 'entityType': 'other',
 'sic': '',
 'sicDescription': '',
 'ownerOrg': '',
 'insiderTransactionForOwnerExists': 0,
 'insiderTransactionForIssuerExists': 1,
 'name': 'Cliffwater Enhanced Lending Fund',
 'tickers': [],
 'exchanges': [],
 'ein': '000000000',
 'lei': None,
 'description': '',
 'website': '',
 'investorWebsite': '',
 'category': '',
 'fiscalYearEnd': '0331',
 'stateOfIncorporation': 'DE',
 'stateOfIncorporationDescription': 'DE',
 'addresses': {'mailing': {'street1': 'C/O UMB FUND SERVICES, INC.',
   'street2': '235 WEST GALENA STREET',
   'city': 'MILWAUKEE',
   'stateOrCountry': 'WI',
   'zipCode': '53212',
   'stateOrCountryDescription': 'WI',
   'isForeignLocation': 0,
   'foreignStateTerritory': None,
   'country': None,
   'countryCode': None},
  'business': {'street1': 'C/O UMB FUND SERVICES, INC.',
   'street2': '235 WEST GALENA STREET',
   'city': 'MILWAUKEE',
   'stateOrCountry': 'WI',
   'zipCode': '53212',
   'stateOrCountryDescription': '

In [13]:
print(json_submissions["filings"].keys())
print(json_submissions["filings"]["recent"].keys())

# every key contains the list, each list is of equal length and same index represent same filings
# accessionNumber[0]  filingDate[0]  form[0]  primaryDocument[0]  → all one filing
# accessionNumber[1]  filingDate[1]  form[1]  primaryDocument[1]  → all one filing
print(type(json_submissions["filings"]["recent"]["accessionNumber"]))

# type of forms filed by the company
print(set(json_submissions["filings"]["recent"]["form"]))

dict_keys(['recent', 'files'])
dict_keys(['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'isXBRLNumeric', 'primaryDocument', 'primaryDocDescription'])
<class 'list'>
{'24F-2NT', 'N-CSR', 'NPORT-P', 'N-CEN', 'CORRESP', '40-17G/A', 'N-PX', '40-17G', '4', 'N-2', 'EFFECT', 'N-2/A', 'N-23C3A', '486APOS', '4/A', 'SC 13D', '40-APP', '3', 'N-CSRS', '424B3', '40-APP/A', 'N-8A', 'DEL AM', 'SC 13G', 'SC 13G/A', '486BPOS'}


In [14]:
# Find indices where form is one of the required forms
target_forms = ["N-CSR", "N-CSRS"]
# target_forms = ["10-K", "10-Q"]
json_submissions_recent = json_submissions["filings"]["recent"]

indices_target_forms = []
for i, form in enumerate(json_submissions_recent["form"]):
    if form in target_forms:
        indices_target_forms.append(i)

print("Found at indices:", indices_target_forms)
print("Total found:", len(indices_target_forms))

Found at indices: [3, 10, 25, 35, 47, 52, 66, 71, 84, 92]
Total found: 10


In [15]:
# the filings are sorted by filing date in descending order
for idx in indices_target_forms:
    print(
        json_submissions_recent["form"][idx],
        "|",
        json_submissions_recent["filingDate"][idx],
        "|",
        json_submissions_recent["reportDate"][idx],
        "|",
        json_submissions_recent["accessionNumber"][idx],
        "|",
        json_submissions_recent["primaryDocument"][idx]
    )

N-CSR | 2026-06-08 | 2026-03-31 | 0001213900-26-066323 | ea0290403-01_ncsr.htm
N-CSRS | 2025-12-05 | 2025-09-30 | 0001213900-25-118509 | ea0264802-01_ncsrs.htm
N-CSR | 2025-06-09 | 2025-03-31 | 0001213900-25-052549 | ea0241375-01_ncsr.htm
N-CSRS | 2024-12-09 | 2024-09-30 | 0001213900-24-106844 | ea0220876-01_ncsrs.htm
N-CSR | 2024-06-07 | 2024-03-31 | 0001213900-24-050670 | ea0205199-01_ncsr.htm
N-CSRS | 2023-12-08 | 2023-09-30 | 0001213900-23-094292 | ea165252_ncsrs.htm
N-CSR | 2023-06-09 |  | 0001213900-23-047761 | ea154416_ncsr.htm
N-CSRS | 2022-12-09 | 2022-09-30 | 0001398344-22-024397 | fp0080845-2_ncsrs.htm
N-CSR | 2022-06-09 | 2022-03-31 | 0001398344-22-011833 | fp0076916_ncsr.htm
N-CSRS | 2021-12-09 | 2021-09-30 | 0001398344-21-023692 | fp0070618_ncsrs.htm


# Download the filing

## 1. Single Filing

In [18]:
# accession = "0001213900-26-066324"
accession = '0001398344-26-004777'
parts = accession.split("-")

print("Filer ID:", parts[0]) # who submitted it (filing agent or company)
print("Year:", parts[1]) # year of filing
print("Sequence:", parts[2]) # sequential number that year

Filer ID: 0001398344
Year: 26
Sequence: 004777


In [19]:
accession_clean = accession.replace("-", "")
cik_plain = str(int(company_cik))  # remove leading zeros for archive URL

print("Clean accession:", accession_clean)
print("Plain CIK:", cik_plain)

Clean accession: 000139834426004777
Plain CIK: 1504234


### 1a. Finding the primary doc from the listing directory

In [20]:
url_filing = f"https://www.sec.gov/Archives/edgar/data/{cik_plain}/{accession_clean}/index.json"
resp_filing = requests.get(url=url_filing, headers=headers)
json_filing = resp_filing.json()
resp_filing.text

'{"directory":{"item":[{"last-modified": "2026-03-06 16:42:02","name":"0001398344-26-004777-index-headers.html","type":"text.gif","size":""},{"last-modified": "2026-03-06 16:42:02","name":"0001398344-26-004777-index.html","type":"text.gif","size":""},{"last-modified": "2026-03-06 16:42:02","name":"0001398344-26-004777.txt","type":"text.gif","size":""},{"last-modified": "2026-03-06 16:42:02","name":"0001398344-26-004777-xbrl.zip","type":"compressed.gif","size":"1046295"},{"last-modified": "2026-03-06 16:42:02","name":"bgx-20251231.xsd","type":"text.gif","size":"10794"},{"last-modified": "2026-03-06 16:42:02","name":"bgx-20251231_def.xml","type":"text.gif","size":"17202"},{"last-modified": "2026-03-06 16:42:02","name":"bgx-20251231_lab.xml","type":"text.gif","size":"24064"},{"last-modified": "2026-03-06 16:42:02","name":"bgx-20251231_pre.xml","type":"text.gif","size":"16656"},{"last-modified": "2026-03-06 16:42:02","name":"FilingSummary.xml","type":"text.gif","size":"2281"},{"last-modifi

In [21]:
print(json_filing["directory"].keys())
print(json_filing["directory"]["item"])
print(json_filing["directory"]["name"])
print(json_filing["directory"]["parent-dir"])

dict_keys(['item', 'name', 'parent-dir'])
[{'last-modified': '2026-03-06 16:42:02', 'name': '0001398344-26-004777-index-headers.html', 'type': 'text.gif', 'size': ''}, {'last-modified': '2026-03-06 16:42:02', 'name': '0001398344-26-004777-index.html', 'type': 'text.gif', 'size': ''}, {'last-modified': '2026-03-06 16:42:02', 'name': '0001398344-26-004777.txt', 'type': 'text.gif', 'size': ''}, {'last-modified': '2026-03-06 16:42:02', 'name': '0001398344-26-004777-xbrl.zip', 'type': 'compressed.gif', 'size': '1046295'}, {'last-modified': '2026-03-06 16:42:02', 'name': 'bgx-20251231.xsd', 'type': 'text.gif', 'size': '10794'}, {'last-modified': '2026-03-06 16:42:02', 'name': 'bgx-20251231_def.xml', 'type': 'text.gif', 'size': '17202'}, {'last-modified': '2026-03-06 16:42:02', 'name': 'bgx-20251231_lab.xml', 'type': 'text.gif', 'size': '24064'}, {'last-modified': '2026-03-06 16:42:02', 'name': 'bgx-20251231_pre.xml', 'type': 'text.gif', 'size': '16656'}, {'last-modified': '2026-03-06 16:42:0

In [22]:
# here primary doc is oklo-20260331.htm
for file in json_filing["directory"]["item"]:
    print(file["name"], "|", file["type"], "|", file["size"])

0001398344-26-004777-index-headers.html | text.gif | 
0001398344-26-004777-index.html | text.gif | 
0001398344-26-004777.txt | text.gif | 
0001398344-26-004777-xbrl.zip | compressed.gif | 1046295
bgx-20251231.xsd | text.gif | 10794
bgx-20251231_def.xml | text.gif | 17202
bgx-20251231_lab.xml | text.gif | 24064
bgx-20251231_pre.xml | text.gif | 16656
FilingSummary.xml | text.gif | 2281
fp0097125-1_01.jpg | image2.gif | 154079
fp0097125-1_02.jpg | image2.gif | 69569
fp0097125-1_03.jpg | image2.gif | 35425
fp0097125-1_04.jpg | image2.gif | 18959
fp0097125-1_05.jpg | image2.gif | 74847
fp0097125-1_06.jpg | image2.gif | 37089
fp0097125-1_07.jpg | image2.gif | 19749
fp0097125-1_08.jpg | image2.gif | 73255
fp0097125-1_09.jpg | image2.gif | 27958
fp0097125-1_10.jpg | image2.gif | 19077
fp0097125-1_11.jpg | image2.gif | 266315
fp0097125-1_ex9912c.htm | text.gif | 21203
fp0097125-1_ex99906cert.htm | text.gif | 7359
fp0097125-1_ex99cert.htm | text.gif | 17529
fp0097125-1_ex99code.htm | text.gif |

### 1b. Using the submissions data primary doc

In [23]:
primary_doc = json_submissions_recent["primaryDocument"][indices_target_forms[0]]
print(primary_doc)

url_primary_doc = f"https://www.sec.gov/Archives/edgar/data/{cik_plain}/{accession_clean}/{primary_doc}"

resp_primary_doc = requests.get(url_primary_doc, headers={"User-Agent": "Zain zain@unicusresearch.com"})
print("Status:", resp_primary_doc.status_code)
print("File size (bytes):", len(resp_primary_doc.content))
print("First 500 chars:")
print(resp_primary_doc.text[:500])

fp0097125-1_ncsrixbrl.htm
Status: 200
File size (bytes): 3484613
First 500 chars:
<?xml version='1.0' encoding='ASCII'?>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:xs="http://www.w3.org/2001/XMLSchema-instance" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xbrldt="http://xbrl.org/2005/xbrldt" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2015-02-26" xmlns:ixt-sec="http://w


In [24]:
filepath = Path("/home/cognify/Downloads")
# filepath = Path("../cliffwater_coporate_lending_fund/CCLFX_Shareholder_reports/2025-09-30/filings/")
filepath.mkdir(parents=True, exist_ok=True)

In [25]:
# Save the file
# filename = f"OKLO_{json_submissions_recent['form'][indices_target_forms[0]]}_{json_submissions_recent['filingDate'][indices_target_forms[0]]}.htm"
filename = Path(f"{filepath}/{primary_doc}")

with open(filename, "wb") as f:
    f.write(resp_primary_doc.content)

## 2. Multiple Filings

In [16]:
cik_plain = str(int(company_cik))  # remove leading zeros for archive URL
print("Plain CIK:", cik_plain)

Plain CIK: 1842754


In [17]:
# dir_filings = Path("../stone_ridge_trust5_lendex/LENDEX_Shareholder_reports/2026-02-28/filings")
dir_filings = Path("/home/cognify/Downloads/SEC_Funds/CPCF_Files")
dir_filings.mkdir(parents=True, exist_ok=True)

In [18]:
count = 1
for idx in indices_target_forms:
    print(f"Downloading index: {idx}")

    primary_doc = json_submissions_recent["primaryDocument"][idx]
    accession = json_submissions_recent["accessionNumber"][idx]
    accession_clean = accession.replace("-", "")

    url_primary_doc = f"https://www.sec.gov/Archives/edgar/data/{cik_plain}/{accession_clean}/{primary_doc}"
    print(f"Primary Doc URL: {url_primary_doc}")

    resp_primary_doc = requests.get(url_primary_doc, headers={"User-Agent": "Zain zain@unicusresearch.com"})
    print("Status:", resp_primary_doc.status_code)

    # Save the file
    # filename = f"LENDEX_{json_submissions_recent['form'][idx]}_{json_submissions_recent['reportDate'][idx]}_{count}.htm"
    filename = primary_doc
    filepath = Path(dir_filings/filename)
    with open(filepath, "wb") as f:
        f.write(resp_primary_doc.content)

    count += 1 

Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390026066323/ea0290403-01_ncsr.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390025118509/ea0264802-01_ncsrs.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390025052549/ea0241375-01_ncsr.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390024106844/ea0220876-01_ncsrs.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390024050670/ea0205199-01_ncsr.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390023094292/ea165252_ncsrs.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000121390023047761/ea154416_ncsr.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842754/000139834422024397/fp0080845-2_ncsrs.htm
Status: 200
Primary Doc URL: https://www.sec.gov/Archives/edgar/data/1842